In [ ]:
!pip install -q -U streamlit requests google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 70.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
!rm -rf ThreatLens
!mkdir -p ThreatLens

print("✅ ThreatLens project folder created")

✅ ThreatLens project folder created


In [ ]:
%%writefile ThreatLens/helpers.py

import socket
import ipaddress
from urllib.parse import urlparse, quote
import requests


# ============================================================
# TARGET VALIDATION
# ============================================================

def validate_target(target, target_type):

    if not target:
        return False, "Please enter a target."

    target = target.strip()

    # -------------------------
    # IP ADDRESS
    # -------------------------

    if target_type == "IP":

        try:
            ipaddress.ip_address(target)
            return True, ""

        except ValueError:
            return False, "Invalid IP address."

    # -------------------------
    # DOMAIN
    # -------------------------

    if target_type == "Domain":

        if "://" in target:
            return False, "Enter only the domain, for example: example.com"

        if " " in target:
            return False, "Domain cannot contain spaces."

        if "." not in target:
            return False, "Please enter a valid domain."

        try:
            socket.gethostbyname(target)

        except socket.gaierror:
            # Domain can still exist even if DNS resolution fails
            pass

        return True, ""

    # -------------------------
    # URL
    # -------------------------

    if target_type == "URL":

        try:

            parsed = urlparse(target)

            if parsed.scheme not in ["http", "https"]:
                return False, "URL must start with http:// or https://"

            if not parsed.netloc:
                return False, "Invalid URL."

            return True, ""

        except Exception:
            return False, "Invalid URL."

    return False, "Unknown target type."


# ============================================================
# ALIENVAULT OTX
# ============================================================

def get_otx(target, target_type, api_key):

    result = {
        "source": "AlienVault OTX",
        "status": "error"
    }

    if not api_key:

        result["message"] = "OTX API key not provided."
        return result

    try:

        # -------------------------
        # IP
        # -------------------------

        if target_type == "IP":

            url = (
                "https://otx.alienvault.com/api/v1/"
                "indicators/IPv4/"
                + target
                + "/general"
            )

        # -------------------------
        # DOMAIN
        # -------------------------

        elif target_type == "Domain":

            url = (
                "https://otx.alienvault.com/api/v1/"
                "indicators/domain/"
                + target
                + "/general"
            )

        # -------------------------
        # URL
        # -------------------------

        elif target_type == "URL":

            encoded = quote(target, safe="")

            url = (
                "https://otx.alienvault.com/api/v1/"
                "indicators/url/"
                + encoded
                + "/general"
            )

        else:

            result["message"] = "Unsupported target type."
            return result

        headers = {
            "X-OTX-API-KEY": api_key,
            "Accept": "application/json"
        }

        response = requests.get(
            url,
            headers=headers,
            timeout=20
        )

        # -------------------------
        # SUCCESS
        # -------------------------

        if response.status_code == 200:

            try:
                data = response.json()

            except Exception:
                data = {}

            result["status"] = "success"
            result["data"] = data

            return result

        # -------------------------
        # FORBIDDEN
        # -------------------------

        if response.status_code == 403:

            result["message"] = "OTX API key was rejected."
            return result

        # -------------------------
        # NOT FOUND
        # -------------------------

        if response.status_code == 404:

            result["status"] = "success"
            result["message"] = "No intelligence found."
            result["data"] = {}

            return result

        # -------------------------
        # OTHER HTTP ERRORS
        # -------------------------

        result["message"] = (
            "OTX returned HTTP "
            + str(response.status_code)
        )

        return result

    except requests.exceptions.Timeout:

        result["message"] = "OTX request timed out."
        return result

    except requests.exceptions.RequestException as e:

        result["message"] = "OTX connection error: " + str(e)
        return result

    except Exception as e:

        result["message"] = "Unexpected OTX error: " + str(e)
        return result


# ============================================================
# DNS / HOST INFORMATION
# ============================================================

def get_dns(target, target_type):

    result = {
        "source": "DNS / Host",
        "status": "error"
    }

    try:

        # -------------------------
        # IP ADDRESS
        # -------------------------

        if target_type == "IP":

            try:

                hostname = socket.gethostbyaddr(target)[0]

            except Exception:

                hostname = "Not available"

            result["status"] = "success"
            result["target"] = target
            result["hostname"] = hostname

            return result

        # -------------------------
        # DOMAIN
        # -------------------------

        if target_type == "Domain":

            hostname = target

        # -------------------------
        # URL
        # -------------------------

        elif target_type == "URL":

            parsed = urlparse(target)
            hostname = parsed.hostname

            if not hostname:

                result["message"] = "Could not extract hostname."
                return result

        else:

            result["message"] = "Unsupported target type."
            return result

        # -------------------------
        # RESOLVE HOSTNAME
        # -------------------------

        ip = socket.gethostbyname(hostname)

        result["status"] = "success"
        result["hostname"] = hostname
        result["resolved_ip"] = ip

        return result

    except socket.gaierror:

        result["status"] = "success"
        result["message"] = "Hostname could not be resolved."

        return result

    except Exception as e:

        result["message"] = str(e)
        return result


# ============================================================
# COLLECT INTELLIGENCE
# ============================================================

def collect_intelligence(
    target,
    target_type,
    otx_api_key
):

    results = {}

    # OTX
    results["AlienVault OTX"] = get_otx(
        target,
        target_type,
        otx_api_key
    )

    # DNS
    results["DNS / Host"] = get_dns(
        target,
        target_type
    )

    return results

Writing ThreatLens/helpers.py


In [ ]:
%%writefile ThreatLens/app.py

import streamlit as st
import json
import re

from google import genai

from helpers import (
    validate_target,
    collect_intelligence
)


# ============================================================
# PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="ThreatLens",
    page_icon="🛡️",
    layout="wide"
)


# ============================================================
# CUSTOM CSS
# ============================================================

st.markdown(
    """
    <style>

    .main-title {
        font-size: 42px;
        font-weight: 700;
        margin-bottom: 5px;
    }

    .subtitle {
        font-size: 17px;
        color: #555;
        margin-bottom: 25px;
    }

    .risk-card {
        padding: 20px;
        border-radius: 12px;
        border: 1px solid #ddd;
        background: #ffffff;
        margin-top: 10px;
        margin-bottom: 20px;
    }

    .risk-number {
        font-size: 42px;
        font-weight: 700;
    }

    .section-title {
        font-size: 22px;
        font-weight: 650;
        margin-top: 20px;
    }

    </style>
    """,
    unsafe_allow_html=True
)


# ============================================================
# HEADER
# ============================================================

st.markdown(
    '<div class="main-title">🛡️ ThreatLens</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="subtitle">'
    'Passive AI-powered threat intelligence analysis'
    '</div>',
    unsafe_allow_html=True
)


# ============================================================
# SIDEBAR
# ============================================================

with st.sidebar:

    st.header("⚙️ Settings")

    target_type = st.selectbox(
        "Target Type",
        [
            "IP",
            "Domain",
            "URL"
        ]
    )

    knowledge_level = st.selectbox(
        "Knowledge Level",
        [
            "Beginner",
            "Intermediate",
            "Expert"
        ]
    )

    st.divider()

    st.subheader("🔑 API Keys")

    otx_api_key = st.text_input(
        "AlienVault OTX API Key",
        type="password"
    )

    gemini_api_key = st.text_input(
        "Gemini API Key",
        type="password"
    )

    st.caption(
        "API keys are entered only for this session."
    )


# ============================================================
# TARGET INPUT
# ============================================================

st.subheader("🎯 Target")

if target_type == "IP":

    placeholder = "8.8.8.8"

elif target_type == "Domain":

    placeholder = "example.com"

else:

    placeholder = "https://example.com"


target = st.text_input(
    "Enter target",
    placeholder=placeholder
)


# ============================================================
# GEMINI ANALYSIS
# ============================================================

def analyze_with_gemini(
    target,
    target_type,
    knowledge_level,
    intelligence,
    api_key
):

    client = genai.Client(
        api_key=api_key
    )

    intelligence_json = json.dumps(
        intelligence,
        indent=2,
        ensure_ascii=False
    )

    prompt = f"""
You are ThreatLens AI, a cybersecurity threat-intelligence assistant.

Analyze ONLY the evidence provided below.

Do not invent information.

Target:
{target}

Target Type:
{target_type}

User Knowledge Level:
{knowledge_level}

Collected Intelligence:
{intelligence_json}

Your task is to determine whether the available evidence indicates
SAFE, SUSPICIOUS, MALICIOUS, or UNKNOWN.

Return ONLY valid JSON.

Use exactly this structure:

{{
    "verdict": "UNKNOWN",
    "confidence": "LOW",
    "risk_score": 0,
    "summary": "Short simple explanation.",
    "evidence": [
        "Evidence point 1",
        "Evidence point 2"
    ],
    "risk": "Short explanation of the risk.",
    "recommended_action": "Short practical recommendation."
}}

Allowed verdicts:

SAFE
SUSPICIOUS
MALICIOUS
UNKNOWN

Allowed confidence:

LOW
MEDIUM
HIGH

Risk score:

0-29 = Low Risk
30-69 = Medium Risk
70-100 = High Risk

Important rules:

- Never invent threat intelligence.
- Never invent malware.
- Never invent threat actors.
- Never invent attacks.
- Never invent CVEs.
- Never invent locations.
- Never claim something is malicious without evidence.
- If evidence is insufficient, use UNKNOWN.
- Keep explanations simple.
- risk_score must be an integer between 0 and 100.
- Return JSON only.
- Do not use Markdown.
- Do not use code fences.
"""

    response = client.models.generate_content(
        model="gemini-3.7-flash",
        contents=prompt
    )

    return response.text


# ============================================================
# PARSE GEMINI RESPONSE
# ============================================================

def parse_analysis(text):

    if not text:

        raise ValueError(
            "Gemini returned an empty response."
        )

    text = text.strip()

    # Remove code fences
    text = re.sub(
        r"^```json",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"^```",
        "",
        text
    )

    text = re.sub(
        r"```$",
        "",
        text
    )

    text = text.strip()

    # Try normal JSON
    try:

        return json.loads(text)

    except json.JSONDecodeError:

        # Try to locate JSON object
        start = text.find("{")
        end = text.rfind("}")

        if start == -1 or end == -1:

            raise ValueError(
                "Gemini did not return valid JSON."
            )

        return json.loads(
            text[start:end + 1]
        )


# ============================================================
# NORMALIZE RESULT
# ============================================================

def normalize_result(data):

    verdict = str(
        data.get(
            "verdict",
            "UNKNOWN"
        )
    ).upper()

    if verdict not in [
        "SAFE",
        "SUSPICIOUS",
        "MALICIOUS",
        "UNKNOWN"
    ]:

        verdict = "UNKNOWN"

    confidence = str(
        data.get(
            "confidence",
            "LOW"
        )
    ).upper()

    if confidence not in [
        "LOW",
        "MEDIUM",
        "HIGH"
    ]:

        confidence = "LOW"

    try:

        risk_score = int(
            data.get(
                "risk_score",
                0
            )
        )

    except Exception:

        risk_score = 0

    risk_score = max(
        0,
        min(
            risk_score,
            100
        )
    )

    summary = str(
        data.get(
            "summary",
            "No summary available."
        )
    )

    evidence = data.get(
        "evidence",
        []
    )

    if not isinstance(
        evidence,
        list
    ):

        evidence = [
            str(evidence)
        ]

    evidence = [
        str(item)
        for item in evidence
    ]

    risk = str(
        data.get(
            "risk",
            "No risk information available."
        )
    )

    recommendation = str(
        data.get(
            "recommended_action",
            "No recommendation available."
        )
    )

    return {
        "verdict": verdict,
        "confidence": confidence,
        "risk_score": risk_score,
        "summary": summary,
        "evidence": evidence,
        "risk": risk,
        "recommended_action": recommendation
    }


# ============================================================
# ANALYZE BUTTON
# ============================================================

if st.button(
    "🔍 Analyze Target",
    type="primary",
    use_container_width=True
):

    # --------------------------------------------------------
    # VALIDATE
    # --------------------------------------------------------

    valid, error = validate_target(
        target,
        target_type
    )

    if not valid:

        st.error(
            "❌ " + error
        )

        st.stop()

    # --------------------------------------------------------
    # API KEY CHECK
    # --------------------------------------------------------

    if not otx_api_key:

        st.error(
            "❌ Enter your AlienVault OTX API key."
        )

        st.stop()

    if not gemini_api_key:

        st.error(
            "❌ Enter your Gemini API key."
        )

        st.stop()

    # --------------------------------------------------------
    # COLLECT INTELLIGENCE
    # --------------------------------------------------------

    with st.spinner(
        "🔎 Collecting threat intelligence..."
    ):

        try:

            intelligence = collect_intelligence(
                target.strip(),
                target_type,
                otx_api_key.strip()
            )

        except Exception as e:

            st.error(
                "Intelligence collection failed: "
                + str(e)
            )

            st.stop()

    # --------------------------------------------------------
    # AI ANALYSIS
    # --------------------------------------------------------

    with st.spinner(
        "🤖 AI is analyzing the evidence..."
    ):

        try:

            raw_result = analyze_with_gemini(
                target.strip(),
                target_type,
                knowledge_level,
                intelligence,
                gemini_api_key.strip()
            )

            analysis = parse_analysis(
                raw_result
            )

            analysis = normalize_result(
                analysis
            )

        except Exception as e:

            st.error(
                "❌ AI analysis failed: "
                + str(e)
            )

            st.stop()

    # ========================================================
    # RESULTS
    # ========================================================

    st.divider()

    st.subheader(
        "🛡️ Threat Assessment"
    )

    verdict = analysis["verdict"]
    confidence = analysis["confidence"]
    risk_score = analysis["risk_score"]

    # --------------------------------------------------------
    # VERDICT
    # --------------------------------------------------------

    if verdict == "SAFE":

        verdict_text = "🟢 SAFE"

    elif verdict == "SUSPICIOUS":

        verdict_text = "🟡 SUSPICIOUS"

    elif verdict == "MALICIOUS":

        verdict_text = "🔴 MALICIOUS"

    else:

        verdict_text = "⚪ UNKNOWN"

    # --------------------------------------------------------
    # METRICS
    # --------------------------------------------------------

    col1, col2, col3 = st.columns(3)

    with col1:

        st.metric(
            "Verdict",
            verdict_text
        )

    with col2:

        st.metric(
            "Risk Score",
            f"{risk_score}%"
        )

    with col3:

        st.metric(
            "Confidence",
            confidence
        )

    # --------------------------------------------------------
    # RISK LEVEL
    # --------------------------------------------------------

    st.markdown(
        "### 📊 Risk Level"
    )

    st.progress(
        risk_score / 100
    )

    if risk_score < 30:

        st.success(
            f"🟢 Low Risk — {risk_score}%"
        )

    elif risk_score < 70:

        st.warning(
            f"🟡 Medium Risk — {risk_score}%"
        )

    else:

        st.error(
            f"🔴 High Risk — {risk_score}%"
        )

    # ========================================================
    # AI SUMMARY
    # ========================================================

    st.markdown(
        "### 💡 AI Summary"
    )

    st.info(
        analysis["summary"]
    )

    # ========================================================
    # EVIDENCE
    # ========================================================

    st.markdown(
        "### 🔎 Evidence"
    )

    if analysis["evidence"]:

        for item in analysis["evidence"]:

            st.write(
                "• " + item
            )

    else:

        st.write(
            "No significant evidence was found."
        )

    # ========================================================
    # RISK
    # ========================================================

    st.markdown(
        "### ⚠️ Risk"
    )

    st.write(
        analysis["risk"]
    )

    # ========================================================
    # RECOMMENDED ACTION
    # ========================================================

    st.markdown(
        "### 🛡️ Recommended Action"
    )

    st.success(
        analysis["recommended_action"]
    )

    # ========================================================
    # SOURCES
    # ========================================================

    st.markdown(
        "### 📡 Intelligence Sources"
    )

    otx = intelligence.get(
        "AlienVault OTX",
        {}
    )

    dns = intelligence.get(
        "DNS / Host",
        {}
    )

    source1, source2 = st.columns(2)

    with source1:

        if otx.get("status") == "success":

            st.success(
                "✓ AlienVault OTX"
            )

        else:

            st.warning(
                "⚠ AlienVault OTX unavailable"
            )

    with source2:

        if dns.get("status") == "success":

            st.success(
                "✓ DNS / Host"
            )

        else:

            st.warning(
                "⚠ DNS / Host unavailable"
            )

    # ========================================================
    # TECHNICAL DETAILS
    # ========================================================

    with st.expander(
        "View technical details"
    ):

        st.json(
            intelligence
        )


# ============================================================
# FOOTER
# ============================================================

st.divider()

st.caption(
    "ThreatLens | Passive Threat Intelligence"
)

Writing ThreatLens/app.py


In [ ]:
%%writefile ThreatLens/requirements.txt

streamlit
requests
google-genai

Writing ThreatLens/requirements.txt


In [ ]:
!python -m py_compile ThreatLens/helpers.py
!python -m py_compile ThreatLens/app.py

print("✅ ThreatLens Python syntax is valid")

✅ ThreatLens Python syntax is valid


In [ ]:
!pkill -f streamlit || true
!pkill -f cloudflared || true

!python -m streamlit run ThreatLens/app.py \
    --server.address=127.0.0.1 \
    --server.port=8501 \
    --server.headless=true \
    > /content/threatlens.log 2>&1 &

^C
^C


In [ ]:
import time

time.sleep(7)

print("========== THREATLENS LOG ==========")

!tail -40 /content/threatlens.log

========== THREATLENS LOG ==========


2026-09-09 16:47:13.635 Uvicorn server started on 127.0.0.1:8501

  You can now view your Streamlit app in your browser.

  URL: http://127.0.0.1:8501



In [ ]:
!curl -I http://127.0.0.1:8501

HTTP/1.1 200 OK
date: Wed, 09 Sep 2026 16:47:48 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 7459
last-modified: Wed, 09 Sep 2026 16:45:30 GMT
etag: "d2d5ea71eeb2819647becadfbb22ed26"
cache-control: no-cache



In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O /content/cloudflared

!chmod +x /content/cloudflared

print("✅ Cloudflare installed")

✅ Cloudflare installed


In [ ]:
!pkill -f cloudflared || true

!/content/cloudflared tunnel \
    --url http://127.0.0.1:8501 \
    --no-autoupdate

^C
2026-09-09T16:48:11Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-09T16:48:11Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-09T16:48:19Z INF +--------------------------------------------------------------------------------------------+
2026-09-09T16:48:19Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-09T16:48:19Z INF |  https://dock-temperature-agency-russell.trycloudfl